In [21]:
# SETUP + IMPORTS
# ─────────────────────────────────────────
import os
import re
import json
import warnings

warnings.filterwarnings('ignore')

from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm
# PDF processing
import fitz  # PyMuPDF
import torch
from langchain.text_splitter import RecursiveCharacterTextSplitter
# Embeddings
from sentence_transformers import SentenceTransformer



In [22]:
import chromadb
# PROJECT PATHS
# ─────────────────────────────────────────
PROJECT_ROOT = Path("C:/Users/sudha/techops-intelligence")
os.chdir(PROJECT_ROOT)

PROCESSED  = PROJECT_ROOT / "data/processed"
RAW_PDFS   = PROJECT_ROOT / "data/raw/pdfs"
EMBEDDINGS = PROJECT_ROOT / "data/embeddings"

print("Setup complete")
print(f"\nPDFs available:")
for pdf in sorted(RAW_PDFS.glob("*.pdf")):
    size = pdf.stat().st_size / 1024 / 1024
    print(f"  {pdf.name:55} {size:.1f} MB")


    
print("Loading embedding model...")
embedding_model = SentenceTransformer(
    'sentence-transformers/all-mpnet-base-v2',
    device='cpu'
)
print("Model ready")

chroma_path    = str(EMBEDDINGS / "chroma_db")
client         = chromadb.PersistentClient(path=chroma_path)
knowledge_base = client.get_or_create_collection(
    name     = "knowledge_base",
    metadata = {"hnsw:space": "cosine"}
)

print(f"knowledge_base: {knowledge_base.count():,} docs")


def get_existing_ids(collection) -> set:
    if collection.count() == 0:
        return set()
    return set(collection.get(include=[])['ids'])


BATCH_SIZE = 32
print("Ready")

Setup complete

PDFs available:
  aws_genai_lens.pdf                                      4.9 MB
  aws_well_architected.pdf                                13.5 MB
  building_secure_and_reliable_systems.pdf                8.9 MB
  high_performance_sre.pdf                                3.4 MB
  site_reliability_engineering.pdf                        7.3 MB
  sre_workbook.pdf                                        13.6 MB
Loading embedding model...


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Model ready
knowledge_base: 0 docs
Ready


## 1. PDF Processing Functions
Extract text from PDFs, chunk, and embed
Chunk size: 700 chars, overlap: 100

In [23]:

def extract_pdf_text(pdf_path):
    doc   = fitz.open(str(pdf_path))
    pages = []
    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text("text").strip()
        if len(text) > 10:
            pages.append({
                "page_num": page_num + 1,
                "text"    : text
            })
    doc.close()
    return pages


def clean_pdf_text(text):
    text = re.sub(r'^\d+$', '', text, flags=re.MULTILINE)
    text = re.sub(r'\x0c', '\n', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()


def chunk_text(text, chunk_size=700, overlap=100):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size    = chunk_size,
        chunk_overlap = overlap,
        separators    = ["\n\n", "\n", ". ", " ", ""]
    )
    chunks = splitter.split_text(text)
    return [c for c in chunks if len(c.strip()) > 20]


def process_pdf(pdf_path):
    print(f"Processing: {pdf_path.name}")
    pages  = extract_pdf_text(pdf_path)
    chunks = []
    for page in pages:
        clean  = clean_pdf_text(page['text'])
        splits = chunk_text(clean)
        for j, chunk in enumerate(splits):
            chunks.append({
                "text"    : chunk,
                "page_num": page['page_num'],
                "source"  : pdf_path.name,
                "chunk_id": j
            })
    print(f"  Pages: {len(pages)}  Chunks: {len(chunks)}")
    return chunks


print("PDF functions ready")

PDF functions ready


## 2. Embed All PDFs
6 SRE and AWS books into knowledge_base

In [24]:
existing    = get_existing_ids(knowledge_base)
pdf_results = {}

for pdf_path in sorted(RAW_PDFS.glob("*.pdf")):
    chunks = process_pdf(pdf_path)
    texts, ids, metadatas = [], [], []

    for chunk in chunks:
        doc_id = (
            f"pdf_{pdf_path.stem}_"
            f"{chunk['page_num']}_{chunk['chunk_id']}"
        )
        if doc_id in existing:
            continue

        texts.append(chunk['text'])
        ids.append(doc_id)
        metadatas.append({
            "source"  : chunk['source'],
            "page_num": str(chunk['page_num']),
            "doc_type": "pdf_knowledge"
        })

    if not texts:
        print(f"  {pdf_path.name}: already embedded")
        pdf_results[pdf_path.name] = 0
        continue

    stored = 0
    for i in tqdm(range(0, len(texts), BATCH_SIZE),
                  desc=pdf_path.stem[:30]):
        bt = texts[i:i+BATCH_SIZE]
        bi = ids[i:i+BATCH_SIZE]
        bm = metadatas[i:i+BATCH_SIZE]
        emb = embedding_model.encode(
            bt, show_progress_bar=False
        )
        knowledge_base.upsert(
            documents  = bt,
            embeddings = emb.tolist(),
            metadatas  = bm,
            ids        = bi
        )
        stored += len(bt)

    pdf_results[pdf_path.name] = stored
    print(f"  Stored: {stored} chunks")

print(f"\nknowledge_base total: {knowledge_base.count():,}")

Processing: aws_genai_lens.pdf
  Pages: 261  Chunks: 974


aws_genai_lens: 100%|██████████| 31/31 [02:43<00:00,  5.26s/it]


  Stored: 974 chunks
Processing: aws_well_architected.pdf
  Pages: 1002  Chunks: 3605


aws_well_architected: 100%|██████████| 113/113 [08:40<00:00,  4.60s/it]


  Stored: 3605 chunks
Processing: building_secure_and_reliable_systems.pdf
  Pages: 536  Chunks: 2294


...

  Stored: 853 chunks
Processing: site_reliability_engineering.pdf
  Pages: 601  Chunks: 2055


site_reliability_engineering: 100%|██████████| 65/65 [04:34<00:00,  4.22s/it]


  Stored: 2055 chunks
Processing: sre_workbook.pdf
  Pages: 488  Chunks: 1924


sre_workbook: 100%|██████████| 61/61 [04:25<00:00,  4.35s/it]

  Stored: 1924 chunks

knowledge_base total: 11,705


## 3. Embed FTF Runbooks
100 FinTechFlow-specific runbooks
Symptom → diagnosis → resolution for each incident type

In [25]:
with open(PROCESSED / "ftf_runbooks.json") as f:
    runbooks = json.load(f)

print(f"Loaded: {len(runbooks)} FTF runbooks")

existing = get_existing_ids(knowledge_base)
texts, ids, metadatas = [], [], []

for idx, rb in enumerate(runbooks):
    doc_id = f"ftf_rb_{idx}"
    if doc_id in existing:
        continue

    sym = rb.get('symptoms', [])
    dia = rb.get('diagnosis_steps', [])
    res = rb.get('resolution_steps', [])

    text = (
        f"Runbook: {rb.get('title', '')} "
        f"Category: {rb.get('category', '')} "
        f"Symptoms: {'. '.join(sym) if isinstance(sym, list) else sym} "
        f"Diagnosis: {'. '.join(dia) if isinstance(dia, list) else dia} "
        f"Resolution: {'. '.join(res) if isinstance(res, list) else res} "
        f"Prevention: {rb.get('prevention', '')}"
    ).strip()

    if len(text) < 30:
        continue

    texts.append(text)
    ids.append(doc_id)
    metadatas.append({
        "source"  : "ftf_runbook",
        "category": rb.get('category', 'application'),
        "team"    : rb.get('team', 'platform-sre'),
        "severity": rb.get('severity_when_triggered', 'P2'),
        "doc_type": "runbook"
    })

if texts:
    emb = embedding_model.encode(
        texts, show_progress_bar=True, batch_size=32
    )
    knowledge_base.upsert(
        documents  = texts,
        embeddings = emb.tolist(),
        metadatas  = metadatas,
        ids        = ids
    )
    print(f"Stored: {len(texts)} runbooks")
else:
    print("All runbooks already embedded")

print(f"knowledge_base total: {knowledge_base.count():,}")

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Loaded: 100 FTF runbooks


Batches: 100%|██████████| 4/4 [00:31<00:00,  7.90s/it]


Stored: 100 runbooks
knowledge_base total: 11,805


## 4. Embed FTF SRE QA Pairs
200 question-answer pairs covering all on-call scenarios

In [26]:
with open(PROCESSED / "ftf_sre_qa.json") as f:
    qa_pairs = json.load(f)

print(f"Loaded: {len(qa_pairs)} FTF QA pairs")

existing = get_existing_ids(knowledge_base)
texts, ids, metadatas = [], [], []

for idx, qa in enumerate(qa_pairs):
    doc_id = f"ftf_qa_{idx}"
    if doc_id in existing:
        continue

    cmds = qa.get('key_commands', [])
    text = (
        f"Question: {qa.get('question', '')} "
        f"Answer: {qa.get('answer', '')} "
        f"Commands: {'. '.join(cmds) if isinstance(cmds, list) else cmds} "
        f"Escalation: {qa.get('escalation_threshold', '')}"
    ).strip()

    if len(text) < 30:
        continue

    texts.append(text)
    ids.append(doc_id)
    metadatas.append({
        "source"  : "ftf_sre_qa",
        "category": qa.get('category', 'incident_response'),
        "doc_type": "sre_knowledge"
    })

if texts:
    emb = embedding_model.encode(
        texts, show_progress_bar=True, batch_size=32
    )
    knowledge_base.upsert(
        documents  = texts,
        embeddings = emb.tolist(),
        metadatas  = metadatas,
        ids        = ids
    )
    print(f"Stored: {len(texts)} QA pairs")
else:
    print("All QA pairs already embedded")

print(f"knowledge_base total: {knowledge_base.count():,}")

Loaded: 170 FTF QA pairs


Batches: 100%|██████████| 6/6 [00:38<00:00,  6.43s/it]

Stored: 170 QA pairs
knowledge_base total: 11,975


In [27]:
def kb_search(query, n=2):
    emb = embedding_model.encode([query]).tolist()
    r   = knowledge_base.query(
        query_embeddings = emb,
        n_results        = n,
        include          = ['documents', 'metadatas', 'distances']
    )
    return r


test_queries = [
    "how to handle on-call escalation P1 incident",
    "PostgreSQL runbook connection refused steps",
    "error budget SLO reliability FinTechFlow",
    "how to unseal HashiCorp Vault at FinTechFlow",
    "cascading failure circuit breaker pattern"
]

print("knowledge_base retrieval test\n")
scores = []
for query in test_queries:
    r     = kb_search(query, n=1)
    score = 1 - r['distances'][0][0]
    src   = r['metadatas'][0][0].get('source', 'unknown')
    text  = r['documents'][0][0][:100]
    scores.append(score)
    print(f"Query  : {query}")
    print(f"Score  : {score:.3f}  Source: {src}")
    print(f"Result : {text}")
    print()

print(f"Avg score: {np.mean(scores):.3f}")
print(f"Min score: {min(scores):.3f}")
print(f"Max score: {max(scores):.3f}")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


knowledge_base retrieval test

Query  : how to handle on-call escalation P1 incident
Score  : 0.660  Source: aws_well_architected.pdf
Result : 2. Set up on-call schedules: Create on-call schedules in Incident Manager that align with your 
esca

Query  : PostgreSQL runbook connection refused steps
Score  : 0.565  Source: ftf_runbook
Result : Runbook: PostgreSQL backup job failing silently Category: database Symptoms: specific symptom 1 as s

Query  : error budget SLO reliability FinTechFlow
Score  : 0.692  Source: high_performance_sre.pdf
Result : is expensive. In addition to engineering hours, there are also opportunities
missed. For instance, y

Query  : how to unseal HashiCorp Vault at FinTechFlow
Score  : 0.900  Source: ftf_sre_qa
Result : Question: How do I unseal HashiCorp Vault at FinTechFlow? Answer: To unseal HashiCorp Vault at FinTe

Query  : cascading failure circuit breaker pattern
Score  : 0.475  Source: site_reliability_engineering.pdf
Result : considerable subset) to fail

In [28]:
print("=" * 50)
print("NOTEBOOK 03 - PDF PIPELINE COMPLETE")
print("=" * 50)
for pdf, count in pdf_results.items():
    print(f"  {pdf:50} {count:4} chunks")
print(f"\n  knowledge_base total : {knowledge_base.count():,} docs")
print(f"  Avg retrieval score  : {np.mean(scores):.3f}")

NOTEBOOK 03 - PDF PIPELINE COMPLETE
  aws_genai_lens.pdf                                  974 chunks
  aws_well_architected.pdf                           3605 chunks
  building_secure_and_reliable_systems.pdf           2294 chunks
  high_performance_sre.pdf                            853 chunks
  site_reliability_engineering.pdf                   2055 chunks
  sre_workbook.pdf                                   1924 chunks

  knowledge_base total : 11,975 docs
  Avg retrieval score  : 0.658
